In [1]:
import pandas as pd
import numpy as np
import requests
import sys

In [2]:
file_path = "../../admixture/variant_population/snp_frequency_variant.tsv"
df = pd.read_csv(file_path, delimiter=' ')

In [182]:
import numpy as np  # Importing NumPy for np.nan
import pandas as pd  # Importing pandas for DataFrame
import requests
import sys

def get_population_frequency(variant):
    
    server = "https://grch37.rest.ensembl.org"
    ext = f"/variation/human/{variant}?pops=1"

    r = requests.get(server+ext, headers={ "Content-Type" : "application/json"})

    if not r.ok:
        r.raise_for_status()
        sys.exit()

    # decode the json object into a python dictionary 
    decoded = r.json()

    # if info not exist 
    if "populations" not in decoded or "name" not in decoded:
        print("Missing expected keys in the JSON response.")
        return None
    
    pop = decoded["populations"]
    
    #Conditional checks for None or Null
    
    combo = [(item.get('population', np.nan),
              item.get('allele', np.nan),
              item.get('frequency', np.nan)) 
             for item in pop 
             if item.get('population', "").startswith('1000GENOMES')]
    
    
    if combo:
        population, allele, frequency = zip(*combo)
        population = list(population)
        frequency = list(frequency)
        allele = list(allele)
    else:
        print(f"No frequency data for variant {variant}")
        population = [item['population'] for item in pop if item['population'].startswith('1000GENOMES')]
        allele = [item['allele'] for item in pop if item['population'].startswith('1000GENOMES')]
        frequency = [np.nan] * len(population)

    ancestry_freq = pd.DataFrame({
        "marker": [decoded["name"]] * len(population),
        "allele": allele,
        "population": population,
        "frequency": frequency
    })
    
    return ancestry_freq


In [184]:
final_df = pd.DataFrame()
for marker in marker_list: 
    temp_df = get_population_frequency(marker)
    final_df = pd.concat([final_df, temp_df], ignore_index = True)

In [185]:
final_df.isna().sum()

marker        0
allele        0
population    0
frequency     0
dtype: int64

In [187]:
final_df.to_csv("../../admixture/variant_population/ensembl_ancestry_allele_frequecy.tsv", sep='\t', index=False)